<a href="https://colab.research.google.com/github/SougataJoy/IO-List-Creation/blob/main/IO_List_Updated006.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================  IO List Generator - v3 (Widgets UI)  ================
# ---- Imports ----
import ipywidgets as widgets
from IPython.display import display, clear_output
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.drawing.image import Image
from openpyxl.utils import get_column_letter, range_boundaries
from google.colab import files
from datetime import datetime
import pytz
import os

# ---- Colab cleanup ----
for f in os.listdir():
    if f.endswith(".png") or f.endswith(".xlsx"):
        os.remove(f)

# ---- Timezone & date ----
ist             = pytz.timezone('Asia/Kolkata')
today_date      = datetime.now(ist).strftime("%Y-%b-%d_%H-%M")
today_date_only = datetime.now(ist).strftime("%d-%m-%Y")

# ---- Style constants ----
THIN  = Side(style='thin')
THICK = Side(style='thick')

YELLOW_FILL = PatternFill(start_color="FFFF00", fill_type="solid")
ORANGE_FILL = PatternFill(start_color="FFC000", fill_type="solid")

TNR_26B = Font(name='Times New Roman', size=26, bold=True)
TNR_9B  = Font(name='Times New Roman', size=9,  bold=True)
CAL_18B = Font(name='Calibri', size=18, bold=True)
CAL_16B = Font(name='Calibri', size=16, bold=True)
CAL_16  = Font(name='Calibri', size=16, bold=False)

CENTER   = Alignment(horizontal='center', vertical='center', wrap_text=True)
CENTER_H = Alignment(horizontal='center', wrap_text=True)
RIGHT    = Alignment(horizontal='right',  wrap_text=True)
LEFT     = Alignment(horizontal='left',   wrap_text=True)

# ==============================================================
#  Classes
# ==============================================================

class Node:
    def __init__(self, number):
        self.number = f"0{number}" if number < 10 else str(number)

# ==============================================================
#  Utility helpers
# ==============================================================

def apply_border(ws, cell_range, side_style='thin'):
    s = Side(style=side_style)
    min_col, min_row, max_col, max_row = range_boundaries(cell_range)
    for row in range(min_row, max_row + 1):
        for col in range(min_col, max_col + 1):
            b = ws.cell(row=row, column=col).border
            ws.cell(row=row, column=col).border = Border(
                top    = s if row == min_row else b.top,
                bottom = s if row == max_row else b.bottom,
                left   = s if col == min_col else b.left,
                right  = s if col == max_col else b.right,
            )

def merge(ws, start_col, end_col, row):
    ws.merge_cells(start_row=row, start_column=start_col,
                   end_row=row,   end_column=end_col)

def set_cell(ws, row, col, value=None, font=None, align=None):
    cell = ws.cell(row=row, column=col)
    if value is not None: cell.value = value
    if font:              cell.font  = font
    if align:
        cell.alignment = Alignment(
            horizontal=align.horizontal,
            vertical=align.vertical,
            wrap_text=True
        )

def set_cell1(ws, cell, value, merge_with=None):
    if merge_with:
        ws.merge_cells(f'{cell}:{merge_with}')
    ws[cell].value     = value
    ws[cell].font      = TNR_9B
    ws[cell].alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

# ==============================================================
#  IO List column definitions
# ==============================================================

IO_COLS = [
    (1,  "Sl. NO",                                    5),
    (2,  "DCS TAG",                                  16),
    (3,  "DCS Tag. No.",                             20),
    (4,  "TAG DESCRIPTION",                          75),
    (5,  "I/O TYPE",                                 10),
    (6,  "Signal Type",                              12),
    (7,  "RANGE",                                     7),
    (8,  None,                                        7),
    (9,  "Engg Unit",                                 9),
    (10, "Dcs Node / Slot",                           8),
    (11, "DCS Card Address/Ch. No.",                 12),
    (12, "DCS Card CH. TB.",                         10),
    (13, "Panel No.",                                24),
    (14, "DCS Wire No.",                             16),
    (15, "Coil SIDE",                                 5),
    (16, "Relay Board (For DO)",                     22),
    (17, "PF SIDE",                                   5),
    (18, "Ferrule at Panel Main TB",                 16),
    (19, "Panel Main TB No.",                        19),
    (20, "Ferrule at Feeder Side (For MCC Panels)",  29),
]

SINGLE_COL_HEADERS = {
    1:  "Sl. NO",
    2:  "DCS TAG",
    3:  "DCS Tag. No.",
    4:  "TAG DESCRIPTION",
    5:  "I/O TYPE",
    6:  "Signal Type",
    9:  "Engg Unit",
    10: "Dcs Node / Slot",
    11: "DCS Card Address/Ch. No.",
    12: "DCS Card CH. TB.",
    13: "Panel No.",
    14: "DCS Wire No.",
    18: "Ferrule at Panel Main TB ",
    19: "Panel Main TB No.",
    20: "Ferrule at Feeder Side (For MCC Panels)",
}

ROW5_LABELS = {7: "Low",  8: "High"}
ROW6_LABELS = {7: "Min",  8: "Max"}

# ==============================================================
#  Sheet builders
# ==============================================================

def build_coverpage(wb, logo, vals):
    """Build CoverPage sheet from a dict of string values."""
    ws = wb.active
    ws.title = "CoverPage"
    ws.sheet_view.showGridLines = False

    for col in range(1, 41):
        ws.column_dimensions[get_column_letter(col)].width = 5.7
    for row in range(1, 50):
        ws.row_dimensions[row].height = 25

    ws.add_image(logo, "B2")
    logo.width, logo.height = 84, 69

    ws.merge_cells('C6:T6')
    ws.merge_cells('U6:W6')
    ws.merge_cells('X6:AI6')
    ws.merge_cells('AJ6:AK6')
    ws.merge_cells('AM6:AN6')

    for row in range(7, 17):
        merge(ws, 3,  20, row)
        merge(ws, 21, 23, row)
        merge(ws, 24, 40, row)

    ws['AJ6'].font = CAL_18B; ws['AJ6'].alignment = RIGHT
    ws['AL6'].font = CAL_18B; ws['AL6'].alignment = CENTER_H
    ws['AM6'].font = CAL_16;  ws['AM6'].alignment = LEFT

    for row in range(6, 17):
        set_cell(ws, row, 3,  font=CAL_18B, align=RIGHT)
        set_cell(ws, row, 21, font=CAL_16B, align=CENTER_H)
        ws[f'X{row}'].font      = CAL_16
        ws[f'X{row}'].alignment = LEFT
        apply_border(ws, f"C{row}:AN{row}")

    apply_border(ws, "AJ6:AN6")

    blocks        = [(3,6),(7,14),(15,22),(23,28),(29,34),(35,40)]
    border_ranges = ["C","G","O","W","AC","AI"]
    border_ends   = ["F","N","V","AB","AH","AN"]
    for row in range(23, 29):
        for sc, ec in blocks:
            ws.merge_cells(f'{get_column_letter(sc)}{row}:{get_column_letter(ec)}{row}')
        for s, e in zip(border_ranges, border_ends):
            apply_border(ws, f"{s}{row}:{e}{row}")
    apply_border(ws, "C23:AN28", side_style='thick')

    label_map = {
        6:  "DWG/DOC TITLE",
        7:  "PROJECT NAME",
        8:  "CLIENT",
        9:  "END USER",
        10: "VENDOR",
        11: "PO REFERENCE",
        12: "JC NO",
        13: "PACKAGE NAME",
        14: "REFERENCE CLIENT DOC. NAME / NO",
        15: "REFERENCE VENDOR DOCUMENT NO",
        16: "CUSTOMER PROJECT CODE",
    }
    for row, text in label_map.items():
        ws[f'C{row}'] = text

    for row in range(6, 17):
        ws[f'U{row}'] = ":"

    ws['X6']  = "IO List"
    ws['AJ6'] = "REV."
    ws['AL6'] = ":"
    ws['AM6'] = "R0"
    ws['X10'] = "M/s. Subtleweigh Electric (I) Pvt. Ltd."

    # ---- fill user values from dict ----
    ws['X7']  = vals['project_name']
    ws['X8']  = f"M/s. {vals['client_name']}"
    ws['X9']  = f"M/s. {vals['end_user']}"
    ws['X11'] = vals['po_ref']
    ws['X12'] = vals['jc_no']
    ws['X13'] = vals['package_name']
    ws['X14'] = vals['ref_client_doc']
    ws['X15'] = vals['ref_vendor_doc']
    ws['X16'] = vals['cust_proj_code']

    rev_data = {
        'C28': "REV. NO.", 'G28': "DATE",         'O28': "DESCRIPTION ",
        'W28': "PREPARED BY", 'AC28': "CHECKED BY", 'AI28': "APPROVED BY",
        'C27': "R0",      'G27': today_date_only, 'O27': "FOR APPROVAL",
        'W27': "SP",      'AC27': "SR",            'AI27': "PKR",
    }
    for addr, val in rev_data.items():
        ws[addr] = val

    rev_cols = ['C', 'G', 'O', 'W', 'AC', 'AI']
    for row, size, bold in [(28, 18, True), (27, 16, False)]:
        for col in rev_cols:
            ws[f'{col}{row}'].font      = Font(name='Calibri', size=size, bold=bold)
            ws[f'{col}{row}'].alignment = CENTER_H


def build_io_list_sheet(wb, sheet_name, vals, node_first, node_last):
    """Create and format one IO List sheet."""
    ws = wb.create_sheet(title=sheet_name)

    for col, _, width in IO_COLS:
        ws.column_dimensions[get_column_letter(col)].width = width

    fills  = [YELLOW_FILL, YELLOW_FILL, ORANGE_FILL]
    titles = [
        vals['end_user'],
        f"{sheet_name} ; {vals['project_name']}",
        f"{vals['po_ref']} / {vals['jc_no']}",
    ]
    for row in range(1, 4):
        ws[f"A{row}"].fill      = fills[row - 1]
        ws[f"A{row}"].value     = titles[row - 1]
        ws[f"A{row}"].font      = TNR_26B
        ws[f"A{row}"].alignment = CENTER
        ws.row_dimensions[row].height = 32
        merge(ws, 1, 20, row)
        apply_border(ws, f"A{row}:T{row}")

    hdr_font  = TNR_9B
    hdr_align = CENTER

    for col, label, _ in IO_COLS:
        cl = get_column_letter(col)
        if col in SINGLE_COL_HEADERS:
            ws.merge_cells(start_row=4, start_column=col, end_row=6, end_column=col)
            cell = ws.cell(row=4, column=col)
            cell.value     = SINGLE_COL_HEADERS[col]
            cell.font      = hdr_font
            cell.alignment = hdr_align
            apply_border(ws, f"{cl}4:{cl}6")
        else:
            if col == 7:
                ws.merge_cells(start_row=4, start_column=7, end_row=4, end_column=8)
                ws.cell(row=4, column=7).value     = "RANGE"
                ws.cell(row=4, column=7).font      = hdr_font
                ws.cell(row=4, column=7).alignment = hdr_align
            if col in ROW5_LABELS:
                ws.cell(row=5, column=col).value     = ROW5_LABELS[col]
                ws.cell(row=5, column=col).font      = hdr_font
                ws.cell(row=5, column=col).alignment = hdr_align
            if col in ROW6_LABELS:
                ws.cell(row=6, column=col).value     = ROW6_LABELS[col]
                ws.cell(row=6, column=col).font      = hdr_font
                ws.cell(row=6, column=col).alignment = hdr_align

    set_cell1(ws, 'O4', "Coil Side")
    set_cell1(ws, 'O5', "PF Side",              merge_with='O6')
    set_cell1(ws, 'P4', "Relay Board (For DO)")
    set_cell1(ws, 'P5', "Relay Board (For DI)", merge_with='P6')
    set_cell1(ws, 'Q4', "PF Side")
    set_cell1(ws, 'Q5', "Coil Side",            merge_with='Q6')

    apply_border(ws, "A4:T6")
    for rng in ["G4:H4","G5:G5","H5:H5","G6:G6","H6:H6",
                "O4:O4","O5:O6","P4:P4","P5:P6","Q4:Q4","Q5:Q6"]:
        apply_border(ws, rng)

    # Node rows (data area — columns pre-formatted, no data yet)
    n = node_first
    while n <= node_last:
        node = Node(n)
        n += 1

    return ws

# ==============================================================
#  Widget layout helpers
# ==============================================================

W = widgets.Layout(width='420px')
WS = widgets.Layout(width='200px')

def labeled(label, widget):
    return widgets.HBox([
        widgets.Label(label, layout=widgets.Layout(width='220px')),
        widget
    ])

# ==============================================================
#  Build the form UI
# ==============================================================

# -- Cover Page fields --
w_project    = widgets.Text(placeholder="e.g. REFINERY EXPANSION",    layout=W)
w_client     = widgets.Text(placeholder="e.g. ABC INDUSTRIES LTD",    layout=W)
w_end_user   = widgets.Text(placeholder="e.g. XYZ CORPORATION",       layout=W)
w_po_ref     = widgets.Text(placeholder="e.g. PO-2024-001",           layout=W)
w_jc_no      = widgets.Text(placeholder="e.g. JC-100",                layout=W)
w_pkg        = widgets.Text(placeholder="e.g. DCS PACKAGE",           layout=W)
w_ref_client = widgets.Text(placeholder="e.g. CLIENT-DOC-001",        layout=W)
w_ref_vendor = widgets.Text(placeholder="e.g. VENDOR-DOC-001",        layout=W)
w_cust_code  = widgets.Text(placeholder="e.g. CUST-PROJ-2024",        layout=W)

# -- Number of IO List sheets --
w_qty = widgets.BoundedIntText(value=1, min=1, max=20,
                               layout=widgets.Layout(width='80px'))

# -- Dynamic per-sheet widgets container --
sheets_box = widgets.VBox([])

# -- Status / log output --
out = widgets.Output()

# ---- Update sheet rows when qty changes ----
def update_sheet_rows(change):
    qty = w_qty.value
    rows = []
    for i in range(qty):
        sn = widgets.Text(value=f"IO List {i+1}", layout=widgets.Layout(width='160px'))
        nf = widgets.BoundedIntText(value=1, min=0, max=999, layout=WS)
        nl = widgets.BoundedIntText(value=5, min=0, max=999, layout=WS)
        row = widgets.HBox([
            widgets.Label(f"Sheet {i+1}:", layout=widgets.Layout(width='60px')),
            widgets.Label("Name:",         layout=widgets.Layout(width='45px')), sn,
            widgets.Label("  Node First:", layout=widgets.Layout(width='85px')), nf,
            widgets.Label("  Node Last:",  layout=widgets.Layout(width='80px')), nl,
        ])
        rows.append(row)
    sheets_box.children = rows

w_qty.observe(update_sheet_rows, names='value')
update_sheet_rows(None)   # initialise with default qty=1

# ---- Generate button ----
btn_generate = widgets.Button(
    description='Generate Excel',
    button_style='success',
    icon='download',
    layout=widgets.Layout(width='200px', height='40px')
)

def on_generate(b):
    out.clear_output()
    with out:
        # -- Collect & validate cover-page values --
        def uv(w):   # uppercase + strip, default N/A
            return w.value.strip().upper() or "N/A"

        vals = {
            'project_name':  uv(w_project),
            'client_name':   uv(w_client),
            'end_user':      uv(w_end_user),
            'po_ref':        uv(w_po_ref),
            'jc_no':         uv(w_jc_no),
            'package_name':  uv(w_pkg),
            'ref_client_doc':uv(w_ref_client),
            'ref_vendor_doc':uv(w_ref_vendor),
            'cust_proj_code':uv(w_cust_code),
        }

        # -- Collect sheet configs --
        sheet_configs = []
        for row_hbox in sheets_box.children:
            children = row_hbox.children
            sn_widget = children[2]   # Text  – sheet name
            nf_widget = children[4]   # BoundedIntText – node first
            nl_widget = children[6]   # BoundedIntText – node last
            name = sn_widget.value.strip() or f"IO List {len(sheet_configs)+1}"
            nf   = nf_widget.value
            nl   = nl_widget.value
            if nl < nf:
                print(f"⚠️  Sheet '{name}': Last Node ({nl}) < First Node ({nf}). Swapping.")
                nf, nl = nl, nf
            sheet_configs.append((name, nf, nl))

        # -- Upload logo --
        print("📂 Please upload the logo file…")
        uploaded  = files.upload()
        if not uploaded:
            print("❌ No file uploaded. Aborting.")
            return
        logo_file = list(uploaded.keys())[0]
        logo      = Image(logo_file)

        # -- Build workbook --
        wb = Workbook()
        print("🔧 Building CoverPage…")
        build_coverpage(wb, logo, vals)

        used_names = set(wb.sheetnames)
        for idx, (name, nf, nl) in enumerate(sheet_configs, 1):
            # ensure unique sheet name
            final_name = name
            suffix = 1
            while final_name in used_names:
                final_name = f"{name}_{suffix}"
                suffix += 1
            used_names.add(final_name)
            print(f"🔧 Building sheet {idx}/{len(sheet_configs)}: '{final_name}' (Nodes {nf}–{nl})…")
            build_io_list_sheet(wb, final_name, vals, nf, nl)

        file_name = f"{today_date}_IO_List.xlsx"
        wb.save(file_name)
        files.download(file_name)
        print(f"\n✅  Saved & downloaded: {file_name}")

btn_generate.on_click(on_generate)

# ==============================================================
#  Render UI
# ==============================================================

header_style = widgets.HTML("<h3 style='margin:8px 0 4px'>📋 IO List Generator</h3>")

cover_section = widgets.VBox([
    widgets.HTML("<b>── Cover Page Details ──</b>"),
    labeled("Project Name",              w_project),
    labeled("Client Name",               w_client),
    labeled("End User",                  w_end_user),
    labeled("PO Reference No.",          w_po_ref),
    labeled("JC No.",                    w_jc_no),
    labeled("Package Name",              w_pkg),
    labeled("Reference Client Doc.",     w_ref_client),
    labeled("Reference Vendor Doc.",     w_ref_vendor),
    labeled("Customer Project Code",     w_cust_code),
], layout=widgets.Layout(margin='0 0 12px 0'))

io_section = widgets.VBox([
    widgets.HTML("<b>── IO List Sheets ──</b>"),
    labeled("Number of IO List sheets:", w_qty),
    widgets.HTML("<i style='color:gray;font-size:12px'>Configure each sheet below:</i>"),
    sheets_box,
], layout=widgets.Layout(margin='0 0 12px 0'))

display(widgets.VBox([
    header_style,
    cover_section,
    io_section,
    btn_generate,
    out,
]))